## Synthetic Patient Records Generation

This notebook includes the generation of the synthetic patient records, derived from expert knowledge and clinical guidelines. The generated records are created to support the development of Clinical Decision Support Systems without reliance on real patient health information.

The process includes five key stages:
1. Reading of the data specification file
2. Random selection of values per clinical feature
3. Clinical Consistency Validation
4. Building Patient Scenarios
5. Dataset Construction

In [ ]:
import json
import random
from pydantic import BaseModel, Field, model_validator
import pandas as pd

#### 1. Reading the JSON feature specification

The generation of the synthetic patient records relies on features and variables defined within a data specification JSON file. This defines the following:
- Core symptoms and their associated weights. Each symptom is mapped to a unique ID, fully specified name, ontology mapping (SNOMED CT) and variables defining whether the presentation is active.
- Risk Factors and prevalence. Similar to symptoms, each risk factor contains a unique ID, fully specified name, mapping to SNOMED CT and activity variables or thresholds. 
- Vital signs and associated range of measurements
- Symptom and Risk Factor Assessment. All symptoms and selected risk factors contain an assessment extension, which represents conditional details associated with the core feature, such as temporal pattern. Each factor within the conditional assessment is defined by presentation variables, describing how the patient may manifest the symptom or risk factor.
- Dependency gating which explains that when a symptom or risk factor is not present, the associated assessment should not be asked

In [ ]:
#Import and read feature specification JSON file
with open ('feature_spec.json', 'r') as file:
    spec_data = json.load (file)

print (json.dumps (spec_data, indent=4))

{
    "dependency_gating": {
        "sentinel_value": "not_asked"
    },
    "symptoms": [
        {
            "SymptomID": "TWC-000",
            "FSN": "Twitching",
            "SCTID": "249966004",
            "Active": {
                "values": [
                    "Yes",
                    "No"
                ],
                "weights": [
                    0.25,
                    0.75
                ]
            },
            "SymptomAssessment": {
                "attributes": [
                    {
                        "AssessmentID": "TWC-001",
                        "FSN": "Symptom Evolution",
                        "Active": {
                            "values": [
                                "Worse",
                                "Same as usual",
                                "New symptom"
                            ]
                        }
                    },
                    {
                        "AssessmentID": "TWC-002",
    

#### 2. Random selection of values per clinical feature

This section defines a function to randomly select associated values per clinical feature. Following the values defined to reflect how a symptom, risk factor or vital might be experienced by a patient, the function checks the mapping between values and core features. Values are selected randomly based on weights defined in the specification file.

In [ ]:
# Define random, weighted selection function
def select_value(variable):
    #Loop through the different values that a feature might have
    for value_type in ["Active", "Threshold_Measurement", "Vital_Measurement"]: 
        if value_type in variable: #Check if the variable actually has values
            value_entry = variable[value_type] #Extract the full feature definition
            
            #For features containing a range (e.g vitals), select ranges according to weights
            if "ranges" in value_entry:
                weights = [r["weight"] for r in value_entry["ranges"]]
                chosen = random.choices(value_entry["ranges"], weights=weights, k=1)[0]
                #Generate value within range
                return random.randint(chosen["min"], chosen["max"])
            
            #Extract weights for values with no range
            values = value_entry["values"]
            weights = value_entry.get("weights", None)
            
            #Generate the random value
            if weights:
                return random.choices(values, weights=weights, k=1)[0]
            else:
                return random.choice (values)
    return None

In [ ]:
#Test random generation of values
test_variable = {"SymptomID": "DRW-000","FSN": "Drowsiness","Active": { "values": ["Yes", "No"], "weights": [0.45, 0.55] }}
print(select_value(test_variable))

#### 3. Clinical Consistency Validation

This section enforces the clinical consistency of patient scenarios prior to generation, as means of ensuring that while records are generated, they align with expectations. Pydantic, a Python library, is used as a key tool for creating a model which represents a complete patient record. The main problem addressed is the implementation of the gating dependency, ensuring that if a symptom or risk factor is not present, their associated assessments are absent as well.

In [ ]:
#Configure patient model
class PatientScenario(BaseModel):
    model_config = {"extra": "allow", "populate_by_name": True}
    
    #Define fixed fields that must be present across all patient records
    patient_ID: str
    OXG_000: int = Field(alias="OXG-000", default=None)
    RSP_000: int = Field(alias="RSP-000", default=None)
    BPM_000: int = Field(alias="BPM-000", default=None)

    #Set up validator to run before patient record generation
    @model_validator(mode="before")
    @classmethod
    def check_gating(cls, data, info):
        #Retrieve specification information from patient model
        spec = info.context.get("spec") if info.context else None
        if not spec:
            return data
        
        #Retrieve sentinel ("not_asked") value
        sentinel = spec["dependency_gating"]["sentinel_value"]
        
        #Iterate through symptoms within patient model and raise error if assessment shows for absent symptom
        for symptom in spec["symptoms"]:
            symptom_ID = symptom["SymptomID"]
            if data.get(symptom_ID) == "No":
                if "SymptomAssessment" in symptom:
                    for attribute in symptom["SymptomAssessment"]["attributes"]:
                        attribute_value = data.get(attribute["AssessmentID"])
                        if attribute_value != sentinel:
                            raise ValueError(
                                f"{attribute['AssessmentID']} should be '{sentinel}' when {symptom_ID} = 'No'"
                            )
        
        #Iterate through risk factors within patient model and raise error if assessment shows for absent risk factor          
        for risk_factor in spec["risk_factors"]:
            risk_factor_ID = risk_factor["RiskFactorID"]
            if data.get(risk_factor_ID) == "No":
                if "RiskFactorAssessment" in risk_factor:
                    for attribute in risk_factor["RiskFactorAssessment"]["attributes"]:
                        attribute_value = data.get(attribute["AssessmentID"])
                        if attribute_value != sentinel:
                            raise ValueError(
                                f"{attribute['AssessmentID']} should be '{sentinel}' when {risk_factor_ID} = 'No'"
                            )
        return data

#### 4. Building patient scenario

This phase builds one complete synthetic patient record. The function takes symptoms, risk factors and vitals from the feature specification file, applies the dependency gating and creates a full patient scenario. The patient scenario is then validated against clinical rules and structure requirements.

In [ ]:
#Define function to build one patient scenario
def build_scenario(spec_data, patient_id):
    #Initialize empty dictionary to hold each patient scenario
    scenario = {} 
    scenario["patient_ID"] = f"ID_{patient_id}"

    #Define dependency gating
    sentinel = spec_data["dependency_gating"]["sentinel_value"]

    #Iterate through symptoms to randomly select values
    for symptom in spec_data["symptoms"]:
        symptom_ID = symptom["SymptomID"] 
        symptom_value = select_value(symptom)
        scenario[symptom_ID]=  symptom_value

        #If the symptom has an assessment and is active, select values for conditional features
        if "SymptomAssessment" in symptom:
            for assessment in symptom["SymptomAssessment"]["attributes"]:
                assessment_ID = assessment["AssessmentID"]
                if symptom_value == "Yes":
                    assessment_value = select_value(assessment)
                    scenario[assessment_ID] = assessment_value
                else:
                    scenario[assessment_ID] = sentinel
    
    #Iterate through vitals to randomly select values from defined ranges
    for vital in spec_data["vitals"]:
        vital_ID = vital["VitalSignID"]
        vital_value = select_value(vital)
        scenario[vital_ID]= vital_value

    #Iterate through risk factors to randomly select values
    for risk_factor in spec_data["risk_factors"]:
        risk_factor_ID = risk_factor["RiskFactorID"]
        risk_factor_value = select_value(risk_factor)
        scenario[risk_factor_ID]= risk_factor_value

        #If the risk factor has an assessment and is active, select values for conditional features
        if "RiskFactorAssessment" in risk_factor:
            for assessment in risk_factor["RiskFactorAssessment"]["attributes"]:
                assessment_ID = assessment["AssessmentID"]
                if risk_factor_value == "Yes":
                    assessment_value = select_value(assessment)
                    scenario[assessment_ID] = assessment_value
                else:
                    scenario[assessment_ID] = sentinel

    #Create narrative symptom presentation summary
    symptom_summary = []
    for symptom in spec_data["symptoms"]:
        symptom_ID = symptom["SymptomID"]
        if scenario.get(symptom_ID) == "Yes":
            symptom_name = symptom["FSN"]
            symptom_details = []
            if "SymptomAssessment" in symptom:
                for attribute in symptom["SymptomAssessment"]["attributes"]:
                    value = scenario.get(attribute["AssessmentID"])
                    if value != sentinel:
                        symptom_details.append(f"{attribute['FSN']}: {value}")
            if symptom_details:
                symptom_summary.append(f"{symptom_name} ({'; '.join(symptom_details)})")
            else:
                symptom_summary.append(f"{symptom_name}")
    
    #Create narrative vital presentation summary
    vital_summary = []
    for vital in spec_data["vitals"]:
        vital_ID = vital["VitalSignID"]
        vital_name = vital["FSN"]
        vital_value = scenario.get(vital_ID)
        vital_summary.append(f"{vital_name}:{vital_value}")

    #Create narrative risk factor presentation summary
    risk_factor_summary = []
    for risk_factor in spec_data["risk_factors"]:
        risk_factor_ID = risk_factor["RiskFactorID"]
        if "Threshold_Measurement" in risk_factor:
            age_value = scenario.get(risk_factor_ID)
            risk_factor_summary.append(f"{risk_factor['FSN']}: {age_value}")
            continue
        if scenario.get(risk_factor_ID) == "Yes":
            risk_factor_name = risk_factor["FSN"]
            risk_factor_summary.append(risk_factor_name)

    #Build text records for symptoms, vitals and risk factor records
    symptom_record = f"The patient is reporting: {', '.join(symptom_summary)}." if symptom_summary else "The patient is not reporting any active symptoms."
    vital_record = f"Current vitals of the patient are: {', '.join(vital_summary)}."
    risk_factor_record = f"Relevant risk factors or comorbidities of the patient include: {', '.join(risk_factor_summary)}." if risk_factor_summary else "No significant risk factors identified."

    #Add all narrative summaries in a structured patient summary
    scenario["Patient_Summary"] = f"{symptom_record} {vital_record} {risk_factor_record}"

    return scenario

#Generate random number for patient ID to identify records
patient_id=random.randint(0, 1000)

#Build patient scenario
patient_scenario= build_scenario(spec_data, patient_id)
print (patient_scenario)

{'patient_ID': 'ID_138', 'TWC-000': 'No', 'TWC-001': 'not_asked', 'TWC-002': 'not_asked', 'TWC-003': 'not_asked', 'TWC-004': 'not_asked', 'HAL-000': 'No', 'HAL-001': 'not_asked', 'HAL-002': 'not_asked', 'HAL-003': 'not_asked', 'DRW-000': 'No', 'DRW-001': 'not_asked', 'DRW-002': 'not_asked', 'SED-000': 'Yes', 'SED-001': 'New symptom', 'SED-002': 'Involuntary sleep episodes', 'SLS-000': 'No', 'SLS-001': 'not_asked', 'SLS-002': 'not_asked', 'SLS-003': 'not_asked', 'SLS-004': 'not_asked', 'AGT-000': 'Yes', 'AGT-001': 'Worse', 'CON-000': 'No', 'CON-001': 'not_asked', 'CON-002': 'not_asked', 'CON-003': 'not_asked', 'CON-004': 'not_asked', 'DIZ-000': 'Yes', 'DIZ-001': 'Worse', 'DIZ-002': 'All the time', 'DIZ-003': 'Vertigo', 'VOM-000': 'No', 'VOM-001': 'not_asked', 'NAU-000': 'No', 'NAU-001': 'not_asked', 'PPL-000': 'No', 'COD-000': 'No', 'COD-001': 'not_asked', 'CYN-000': 'No', 'OXG-000': 90, 'RSP-000': 20, 'BPM-000': 72, 'REN-000': 'No', 'LIV-000': 'No', 'SAP-000': 'No', 'LNG-000': 'No', 'M

In [ ]:
#Validate individual patient scenario
def scenario_validation (patient_scenario, spec_data):
    #Check patient scenario has at least 2 symptoms
    symptom_count = 0
    for symptom in spec_data["symptoms"]:
        symptom_ID = symptom["SymptomID"]
        if patient_scenario.get(symptom_ID) == "Yes":
            symptom_count += 1
    if symptom_count < 2:
        return False
    
    #Check patient scenario has at least one risk factor
    risk_factor_count = 0
    for risk_factor in spec_data["risk_factors"]:
        risk_factor_ID = risk_factor["RiskFactorID"]
        if patient_scenario.get(risk_factor_ID) == "Yes":
            risk_factor_count += 1
    if risk_factor_count < 1:
        return False
    
    #Patients that are opioid naive (never took opioids) can not have a history of toxicity or an increase in dosage
    if patient_scenario.get("MED-OPD-001") == "Yes" and patient_scenario.get("HST-OPX-000") == "Yes":
        return False
    if patient_scenario.get("MED-OPD-001") == "Yes" and patient_scenario.get("MED-OPD-002") == "Yes":
        return False
    return True

#### 5. Dataset Construction

This section generates 350 patient scenarios, validating them against the rules defined and Pydantic's requirements before creating a full dataset with synthetic patient records. 

In [ ]:

random.seed(42)
def data_generation (spec_data, n):
    patient_dataset = []
    used_patient_id = set()
    #Generate random patient_id
    while len(patient_dataset) < n:
        patient_id=random.randint(0, 1000)
        if patient_id in used_patient_id:
            continue
        used_patient_id.add(patient_id)
        #Build patient scenarios and check their validity
        patient_scenario= build_scenario(spec_data, patient_id)
        if scenario_validation (patient_scenario, spec_data):
            try:
                PatientScenario.model_validate(patient_scenario, context={"spec": spec_data})
                patient_dataset.append(patient_scenario)
            except Exception as e:
                print(f"Validation error: {e}")
    return patient_dataset

dataset = data_generation(spec_data, 350)
df = pd.DataFrame(dataset)
display(df)
df.to_csv("dataset.csv", index=False)

,patient_ID,TWC-000,TWC-001,TWC-002,TWC-003,TWC-004,HAL-000,HAL-001,HAL-002,HAL-003,...,MED-REC-000,MED-OPD-001,MED-OPD-002,MED-OPD-002-1,MED-OPD-003,ALC-000,HST-OPX-000,INF-000,AGE-000,Patient_Summary
0,ID_654,Yes,New symptom,At unpredictable times,Trembling,Legs or feet,No,not_asked,not_asked,not_asked,...,No,No,No,not_asked,No,No,No,No,Below 65,The patient is reporting: Twitching (Symptom E...
1,ID_233,No,not_asked,not_asked,not_asked,not_asked,No,not_asked,not_asked,not_asked,...,No,No,No,not_asked,Yes,No,No,No,Below 65,The patient is reporting: Dizziness (Symptom E...
2,ID_135,No,not_asked,not_asked,not_asked,not_asked,No,not_asked,not_asked,not_asked,...,No,No,Yes,In the last 7-14 days,No,No,No,No,Below 65,The patient is reporting: Sedation (Symptom Ev...
3,ID_209,No,not_asked,not_asked,not_asked,not_asked,No,not_asked,not_asked,not_asked,...,No,No,Yes,In the last 7-14 days,No,No,No,No,Below 65,The patient is reporting: Drowsiness (Symptom ...
4,ID_830,Yes,Same as usual,After physical activity,Trembling,Face,No,not_asked,not_asked,not_asked,...,No,No,Yes,In the last 7-14 days,Yes,No,No,No,Below 65,The patient is reporting: Twitching (Symptom E...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
345,ID_515,No,not_asked,not_asked,not_asked,not_asked,No,not_asked,not_asked,not_asked,...,No,No,Yes,In the last 7-14 days,Yes,No,No,No,Below 65,The patient is reporting: Sedation (Symptom Ev...
346,ID_35,No,not_asked,not_asked,not_asked,not_asked,No,not_asked,not_asked,not_asked,...,No,No,No,not_asked,Yes,Yes,No,No,65 or Above,The patient is reporting: Sedation (Symptom Ev...
347,ID_429,Yes,New symptom,After physical activity,Sudden jerks,Legs or feet,No,not_asked,not_asked,not_asked,...,Yes,No,No,not_asked,Yes,No,Yes,No,Below 65,The patient is reporting: Twitching (Symptom E...
348,ID_553,No,not_asked,not_asked,not_asked,not_asked,No,not_asked,not_asked,not_asked,...,No,No,No,not_asked,Yes,Yes,No,No,65 or Above,The patient is reporting: Slurred Speech (Symp...
